# 2. Embed examples into pgvector

Fills the single `examples` table in Lakebase Postgres that the LLM
classifier (notebook 3) retrieves against. For every schema we embed:

* **taxonomy leaves** — cold-start candidates (skipped for very large
  taxonomies like UNSPSC to bound cost);
* **ground-truth invoices** — few-shot examples that get stronger as
  corrections accumulate.

All embedding calls go through **Unity Gateway**. pgvector is also the
app's backend, so there is only ever one store to understand.

This notebook does **not** classify anything — notebook 3 does.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openpyxl pydantic 'databricks-sdk>=0.50.0' psycopg2-binary openai mlflow
%restart_python

In [ ]:
import os, sys, uuid
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
from src.schemas import list_schemas
from src.categorize import embed_examples_into_pgvector
from src.llm import gateway_client

spark = get_spark()
config = load_config()
client = gateway_client()   # Unity Gateway

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('lakebase_instance', config.lakebase_instance)
dbutils.widgets.text('embed_endpoint', 'databricks-gte-large-en')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
lakebase_instance = dbutils.widgets.get('lakebase_instance')
embed_endpoint = dbutils.widgets.get('embed_endpoint')
spark.sql(f'USE {catalog}.{schema}')

## Connect to Lakebase

One psycopg2 connection, reused for every schema. The deployed app's
service principal needs to read the `examples` table — granted at the end.

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg2

w = WorkspaceClient()

def pg_connect():
    inst = w.database.get_database_instance(name=lakebase_instance)
    cred = w.database.generate_database_credential(
        request_id=str(uuid.uuid4()), instance_names=[lakebase_instance]
    )
    me = w.current_user.me()
    user = me.emails[0].value if me.emails else me.user_name
    return psycopg2.connect(
        host=inst.read_write_dns, dbname=config.lakebase_dbname,
        user=user, password=cred.token, sslmode='require',
    )

SP_ROLE = 'aab57292-8e82-48bf-8e43-e8f43fe2743f'  # spend-app service principal
conn = pg_connect()

## Embed every schema into pgvector

For each schema, `embed_examples_into_pgvector` builds the taxonomy +
ground-truth example rows, embeds their text through Unity Gateway, and
replaces that schema's rows in the `examples` table. Re-running is safe —
each schema's rows are deleted and rewritten.

In [ ]:
from src import vectorstore

for s in list_schemas():
    print(f'==> {s.name}')
    embed_examples_into_pgvector(
        spark, s,
        catalog=catalog, schema_db=schema,
        conn=conn, client=client, embed_endpoint=embed_endpoint,
    )

# Let the app's service principal read the examples table.
try:
    vectorstore.grant_select(conn, schema, SP_ROLE)
    print('granted SELECT on examples to the app SP')
except Exception as e:
    print(f'grant skipped: {e}')

conn.close()